# EarthScape Climate Agency — Notebook 01: Data Ingestion

### Objective:
Ingest the primary weather event dataset (`WeatherEvents_Jan2016-Dec2022.csv`), inspect its dimensions, schemas, memory usage, and establish connections to MongoDB and Hadoop HDFS storage.


## 1. Import Ingestion Libraries


In [ ]:
import pandas as pd
import numpy as np
import os
import sys
from pymongo import MongoClient

print("Ingestion environment ready.")


## 2. Load Raw Weather Events Dataset
We load the comprehensive US weather telemetry dataset.


In [ ]:
DATA_PATH = '../WeatherEvents_Jan2016-Dec2022.csv'
print(f"Dataset File Size: {os.path.getsize(DATA_PATH) / (1024*1024):.2f} MB")

# Ingest dataset using chunked / head reading for inspection
df_raw = pd.read_csv(DATA_PATH, nrows=100000)
print(f"Loaded {len(df_raw):,} records for initial inspection.")
df_raw.head()


## 3. Dataset Dimensions & Column Profiling


In [ ]:
print(f"Rows: {df_raw.shape[0]:,}, Columns: {df_raw.shape[1]}")
print("\nColumns in dataset:")
for col in df_raw.columns:
    print(f" - {col}")


## 4. Data Types and Memory Footprint


In [ ]:
df_raw.info()


## 5. Summary Statistics of Numerical Attributes


In [ ]:
df_raw.describe()


## 6. MongoDB Connection & Batch Ingestion
We connect to MongoDB and perform batch insertion (10,000 records per batch) to prevent memory bottlenecks.


In [ ]:
client = MongoClient('mongodb://localhost:27017/')
db = client['earthscape_climate_db']
raw_collection = db['weather_events_raw']

batch_size = 10000
sample_dict = df_raw.head(20000).to_dict(orient='records')

try:
    for i in range(0, len(sample_dict), batch_size):
        batch = sample_dict[i:i + batch_size]
        raw_collection.insert_many(batch)
    print(f"Successfully inserted {raw_collection.count_documents({})} raw records into MongoDB.")
except Exception as e:
    print(f"Ingestion note: {e}")


## 7. Raw Data Path for HDFS Staging


In [ ]:
# Hadoop HDFS Command Demonstration for staging
hdfs_raw_path = "/climate/raw/WeatherEvents_Jan2016-Dec2022.csv"
print(f"HDFS Target Staging URI: {hdfs_raw_path}")
print("Run: hdfs dfs -put ../WeatherEvents_Jan2016-Dec2022.csv /climate/raw/")


### Conclusion:
Data ingestion pipeline successfully loads the CSV dataset, inspects schema and memory footprint, establishes local MongoDB staging, and prepares HDFS paths.
